In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [2]:
data = pd.read_csv("/kaggle/input/datasets/animatronbot/mnist-digit-recognizer/train.csv")

In [3]:
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
data = np.array(data)
m,n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1:n]\

data_Train = data[1000:m].T
Y_train = data_Train[0]
X_train = data_Train[1:n]

In [5]:
def init_params():
    W1 = np.random.rand(10,784) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10,10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def Softmax(Z):
    Z_shifted = Z - np.max(Z, axis=0)
    return np.exp(Z_shifted) / np.sum(np.exp(Z_shifted), axis=0)
    
def forward_prop(W1,b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 =ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = Softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def derev_ReLU(Z):
    return Z>0
    
def back_prop(Z1, A1, Z2, A2, W2, Y, X):
    m = Y.size
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1/m * dZ2 .dot(A1.T)
    db2 = 1/m * np.sum(dZ2,1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * derev_ReLU(Z1)
    dW1 = 1/m * dZ1 .dot(X.T)
    db1 = 1/m * np.sum(dZ1, 1, keepdims=True)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [6]:
X_train = X_train / 255.
print("Max pixel value is now:", np.max(X_train))

Max pixel value is now: 1.0


In [7]:
def loss(X, Y, A2):
    m = Y.size
    log_likelihood = -np.log(A2[Y, range(m)])
    loss = 1/m * np.sum(log_likelihood)
    return loss

def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_decent(X, Y, iterations, alpha):
    W1, b1, W2, b2  = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, Y, X)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 50 == 0:
            print("iterations: ",  i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2        

In [8]:
W1, b1, W2, b2 = gradient_decent(X_train, Y_train, 500, 0.1)

iterations:  0
[0 0 0 ... 0 0 0] [1 9 1 ... 3 3 3]
Accuracy:  0.05314634146341463
iterations:  50
[1 9 1 ... 1 0 3] [1 9 1 ... 3 3 3]
Accuracy:  0.3695609756097561
iterations:  100
[1 9 1 ... 1 0 3] [1 9 1 ... 3 3 3]
Accuracy:  0.5189024390243903
iterations:  150
[1 9 1 ... 1 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.620219512195122
iterations:  200
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.6875365853658536
iterations:  250
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.7313414634146341
iterations:  300
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.7611463414634146
iterations:  350
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.7828048780487805
iterations:  400
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.7994878048780488
iterations:  450
[1 9 1 ... 5 5 3] [1 9 1 ... 3 3 3]
Accuracy:  0.8137317073170731
